In [ ]:
"""
COMPLETE ROSSMANN MODELING PIPELINE
Run all three models and compare results
"""

import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Import our custom modules
from src.models.baseline_models import SeasonalNaiveForecaster, MovingAverageForecaster
from src.models.ml_models import LightGBMForecaster
from src.evaluation.metrics import calculate_metrics, plot_predictions, compare_models

print("="*70)
print("ROSSMANN STORE SALES - COMPLETE MODELING PIPELINE")
print("="*70)

# ============================================================================
# STEP 1: LOAD DATA
# ============================================================================

print("\n📂 STEP 1: Loading data...")
data = pd.read_csv('../data/processed/train_features.csv', parse_dates=['Date'])

print(f"  ✓ Data loaded: {data.shape}")
print(f"  Date range: {data['Date'].min()} to {data['Date'].max()}")
print(f"  Total stores: {data['Store'].nunique()}")

# ============================================================================
# STEP 2: TRAIN-VALIDATION SPLIT
# ============================================================================

print("\n📊 STEP 2: Splitting data...")
split_date = '2015-06-16'

train = data[data['Date'] < split_date].reset_index(drop=True)
valid = data[data['Date'] >= split_date].reset_index(drop=True)

print(f"  Train: {train.shape[0]:,} rows ({train['Date'].min()} to {train['Date'].max()})")
print(f"  Valid: {valid.shape[0]:,} rows ({valid['Date'].min()} to {valid['Date'].max()})")

# ============================================================================
# STEP 3.5: ENCODE CATEGORICALS FOR LGBM

def encode_categoricals(df):
    """Encode categorical columns for LightGBM"""
    df = df.copy()

    # Specific mappings (stable across train/valid)
    if 'StateHoliday' in df.columns:
        state_map = {'0': 0, 'a': 1, 'b': 2, 'c': 3}
        df['StateHoliday'] = df['StateHoliday'].astype(str).map(state_map).fillna(0).astype(int)

    if 'StoreType' in df.columns:
        store_type_map = {'a': 0, 'b': 1, 'c': 2, 'd': 3}
        df['StoreType'] = df['StoreType'].map(store_type_map).fillna(0).astype(int)

    if 'Assortment' in df.columns:
        assortment_map = {'a': 0, 'b': 1, 'c': 2}
        df['Assortment'] = df['Assortment'].map(assortment_map).fillna(0).astype(int)

    # Generic fallback for any remaining object cols (e.g., PromoInterval)
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype('category').cat.codes

    return df

# Apply encoding for LightGBM
train_enc = encode_categoricals(train)
valid_enc = encode_categoricals(valid)

# ============================================================================
# STEP 3: BASELINE MODEL - SEASONAL NAIVE
# ============================================================================

print("\n" + "="*70)
print("MODEL 1: SEASONAL NAIVE BASELINE")
print("="*70)

baseline = SeasonalNaiveForecaster(seasonality=7)
baseline.fit(train)

print("\n🔮 Generating baseline predictions...")
baseline_preds = baseline.predict(valid)

baseline_metrics = calculate_metrics(valid['Sales'].values, baseline_preds.values)

print("\n📊 Baseline Results:")
for metric, value in baseline_metrics.items():
    print(f"  {metric}: {value:.4f}")

# ============================================================================
# STEP 4: LIGHTGBM MODEL
# ============================================================================

print("\n" + "="*70)
print("MODEL 2: LIGHTGBM")
print("="*70)

lgb_model = LightGBMForecaster()
lgb_model.fit(train_enc, valid_enc, num_iterations=1000)

print("\n🔮 Generating LightGBM predictions...")
lgb_preds = lgb_model.predict(valid_enc)

lgb_metrics = calculate_metrics(valid['Sales'].values, lgb_preds)

print("\n📊 LightGBM Results:")
for metric, value in lgb_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Feature importance
print("\n📈 Top 15 Important Features:")
importance_df = lgb_model.get_feature_importance(top_n=15)
print(importance_df.to_string(index=False))

# ============================================================================
# STEP 5: MODEL COMPARISON
# ============================================================================

print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)

results = {
    'Seasonal Naive': {'y_true': valid['Sales'].values, 'y_pred': baseline_preds.values},
    'LightGBM': {'y_true': valid['Sales'].values, 'y_pred': lgb_preds}
}

comparison_df = compare_models(results)
print("\n📊 Model Performance Comparison:")
print(comparison_df.to_string(index=False))

# ============================================================================
# STEP 6: VISUALIZATIONS
# ============================================================================

print("\n📊 Generating visualizations...")

# Plot 1: Model Comparison
fig, ax = plt.subplots(figsize=(10, 6))
comparison_df.plot(x='Model', y='RMSPE', kind='bar', ax=ax, color='steelblue', legend=False)
ax.set_title('Model Performance Comparison (Lower is Better)', fontsize=14, fontweight='bold')
ax.set_ylabel('RMSPE')
ax.set_xlabel('Model')
ax.grid(alpha=0.3, axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Plot 2: LightGBM Predictions
fig = plot_predictions(valid['Sales'].values, lgb_preds, 
                       title='LightGBM Model', sample_size=2000)
plt.show()

# Plot 3: Feature Importance
fig, ax = plt.subplots(figsize=(10, 8))
importance_df.plot(x='feature', y='importance', kind='barh', ax=ax, color='forestgreen', legend=False)
ax.set_title('Top 15 Feature Importance (LightGBM)', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

# ============================================================================
# STEP 7: SAVE BEST MODEL
# ============================================================================

import os

print("\n💾 Saving best model...")

# Create directories if they don't exist
os.makedirs('../models', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

# Save model
try:
    lgb_model.save_model('../models/lightgbm_best_model.pkl')
    print("  ✓ Model saved successfully")
except Exception as e:
    print(f"  ✗ Error saving model: {e}")

# Save predictions
try:
    results_df = pd.DataFrame({
        'Store': valid['Store'],
        'Date': valid['Date'],
        'Actual_Sales': valid['Sales'],
        'Baseline_Pred': baseline_preds.values,
        'LightGBM_Pred': lgb_preds
    })
    
    results_df.to_csv('../outputs/validation_predictions.csv', index=False)
    print("  ✓ Predictions saved to ../outputs/validation_predictions.csv")
except Exception as e:
    print(f"  ✗ Error saving predictions: {e}")

print("\n" + "="*70)
print("✅ MODELING COMPLETE!")
print("="*70)
print(f"\n🏆 Best Model: LightGBM")
print(f"📊 Best RMSPE: {lgb_metrics['RMSPE']:.4f}")
print(f"💾 Model saved: ../models/lightgbm_best_model.pkl")